# Semana 7: Aprendizaje no supervisado y clustering
## Fundamentos de IA y Machine Learning — ULACIT, 1c0-2026

**Objetivo de esta sesion:** Aprender los conceptos fundamentales del aprendizaje no supervisado y clustering ejecutando paso a paso un flujo de trabajo completo, desde la preparacion de datos hasta la interpretacion de resultados para decisiones de negocio.

**Instrucciones:**
- Ejecuten cada celda en orden (Shift + Enter o boton ▶)
- Lean los textos explicativos entre celdas: ahi esta la teoria
- No necesitan modificar codigo salvo donde se indique explicitamente
- Al final hay preguntas de discusion para la sesion

---

## 1. ¿Que es el aprendizaje no supervisado?

En las semanas anteriores trabajamos con **aprendizaje supervisado**: teniamos datos etiquetados (por ejemplo, "este cliente abandono" o "esta transaccion es fraude") y el modelo aprendia a predecir esas etiquetas.

El **aprendizaje no supervisado** es diferente: los datos **no tienen etiquetas**. El algoritmo debe descubrir patrones, estructuras y grupos por si mismo.

### Analogia

Imaginen que llegan a una fiesta donde no conocen a nadie y nadie tiene etiqueta con su nombre. Observando como hablan, como visten, en que grupos se reunen, ustedes naturalmente empiezan a identificar quien podria ser amigo de quien, quienes son colegas, quienes llegaron solos. Eso es aprendizaje no supervisado: **descubrir estructura sin que nadie les diga la respuesta**.

### Diferencias clave

| Aspecto | Supervisado | No supervisado |
|---------|------------|----------------|
| **Datos** | Etiquetados (con respuestas conocidas) | Sin etiquetar |
| **Objetivo** | Predecir resultados especificos | Descubrir patrones ocultos |
| **Evaluacion** | Directa (prediccion vs. realidad) | Compleja (no hay "respuesta correcta") |
| **Ejemplos** | Clasificacion de spam, prediccion de ventas | Segmentacion de clientes, deteccion de anomalias |
| **Costo de datos** | Alto (requiere etiquetar manualmente) | Bajo (usa datos crudos) |

### Aplicaciones principales del aprendizaje no supervisado

1. **Clustering:** agrupar datos similares (segmentacion de clientes, agrupacion de productos)
2. **Reduccion de dimensionalidad:** simplificar datos complejos conservando lo esencial (PCA — lo veremos en la semana 8)
3. **Deteccion de anomalias:** identificar puntos atipicos (fraudes, fallas de equipo)
4. **Reglas de asociacion:** descubrir relaciones ("los clientes que compran X tambien compran Y")

Hoy nos enfocamos en **clustering**, la tecnica mas utilizada en contextos empresariales.

## 2. Preparacion del entorno

Primero importamos las librerias que necesitamos. Todas estan preinstaladas en Google Colab.

In [ ]:
# =============================================================
# Librerias necesarias (todas preinstaladas en Colab)
# =============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap
import seaborn as sns

# Algoritmos de clustering
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

# Metricas de evaluacion
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

# Herramientas auxiliares
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs, make_moons
from scipy.cluster.hierarchy import dendrogram, linkage

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

import warnings
warnings.filterwarnings('ignore')

print("Librerias cargadas correctamente.")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 3. Nuestro escenario: segmentacion de clientes

Vamos a trabajar con un escenario clasico de negocios: **segmentacion de clientes** usando el modelo **RFM** (Recencia, Frecuencia, Valor Monetario).

El modelo RFM es el estandar de la industria para segmentacion y funciona con tres variables:
- **Recencia (R):** hace cuantos dias fue la ultima compra del cliente (menor = mejor)
- **Frecuencia (F):** cuantas compras ha realizado (mayor = mejor)
- **Valor monetario (M):** cuanto dinero ha gastado en total (mayor = mejor)

Vamos a simular datos de 500 clientes de una empresa ficticia de e-commerce.

In [ ]:
# =============================================================
# Generamos datos sinteticos de 500 clientes con perfil RFM
# =============================================================
np.random.seed(42)  # Para que todos obtengamos los mismos resultados

n_clientes = 500

# Simulamos 4 tipos de clientes "ocultos" que el algoritmo debera descubrir:
# Grupo 1: Clientes premium (compran seguido, gastan mucho, compraron recientemente)
# Grupo 2: Clientes leales pero de bajo valor (compran seguido, gastan poco)
# Grupo 3: Clientes nuevos o esporadicos (pocas compras, gasto medio)
# Grupo 4: Clientes inactivos (no compran hace mucho, pocas compras)

# -- Grupo 1: Premium (~100 clientes)
r1 = np.random.normal(15, 8, 100).clip(1, 60)       # Recencia baja (compraron hace poco)
f1 = np.random.normal(25, 5, 100).clip(5, 40)        # Frecuencia alta
m1 = np.random.normal(150000, 30000, 100).clip(50000) # Valor alto (en colones)

# -- Grupo 2: Leales de bajo valor (~130 clientes)
r2 = np.random.normal(30, 12, 130).clip(1, 90)
f2 = np.random.normal(18, 4, 130).clip(5, 30)
m2 = np.random.normal(45000, 12000, 130).clip(10000)

# -- Grupo 3: Esporadicos (~150 clientes)
r3 = np.random.normal(60, 20, 150).clip(10, 150)
f3 = np.random.normal(5, 3, 150).clip(1, 15)
m3 = np.random.normal(65000, 25000, 150).clip(5000)

# -- Grupo 4: Inactivos (~120 clientes)
r4 = np.random.normal(120, 30, 120).clip(30, 250)
f4 = np.random.normal(3, 2, 120).clip(1, 10)
m4 = np.random.normal(30000, 15000, 120).clip(5000)

# Combinamos todo en un DataFrame
df = pd.DataFrame({
    'recencia_dias': np.concatenate([r1, r2, r3, r4]),
    'frecuencia_compras': np.concatenate([f1, f2, f3, f4]),
    'valor_monetario_colones': np.concatenate([m1, m2, m3, m4])
})

# Mezclamos aleatoriamente (el algoritmo NO debe saber los grupos)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.index.name = 'cliente_id'

print(f"Dataset creado: {df.shape[0]} clientes, {df.shape[1]} variables")
print(f"\nPrimeros 10 registros:")
df.head(10)

## 4. Analisis exploratorio antes de clusterizar

Antes de aplicar cualquier algoritmo, **siempre** debemos explorar los datos. Esto es critico porque:
- Nos ayuda a entender la distribucion de las variables
- Nos permite detectar valores atipicos (outliers) que podrian afectar el clustering
- Nos muestra si las variables tienen escalas muy diferentes (spoiler: si las tienen, y eso importa mucho)

In [ ]:
# =============================================================
# Estadisticas descriptivas
# =============================================================
print("Estadisticas descriptivas de nuestros clientes:\n")
df.describe().round(1)

In [ ]:
# =============================================================
# Visualizacion: distribuciones y relaciones entre variables
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogramas de cada variable
variables = ['recencia_dias', 'frecuencia_compras', 'valor_monetario_colones']
titulos = ['Recencia (dias desde ultima compra)', 'Frecuencia (numero de compras)', 'Valor monetario (colones)']
colores = ['#2d1b4e', '#00d4ff', '#7c3aed']

for i, (var, titulo, color) in enumerate(zip(variables, titulos, colores)):
    axes[i].hist(df[var], bins=30, color=color, alpha=0.7, edgecolor='white')
    axes[i].set_title(titulo, fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Cantidad de clientes')

plt.tight_layout()
plt.suptitle('Distribucion de las variables RFM', fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# =============================================================
# Scatter plot: relaciones entre pares de variables
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(df['recencia_dias'], df['frecuencia_compras'], alpha=0.5, s=20, c='#2d1b4e')
axes[0].set_xlabel('Recencia (dias)')
axes[0].set_ylabel('Frecuencia (compras)')
axes[0].set_title('Recencia vs Frecuencia')

axes[1].scatter(df['frecuencia_compras'], df['valor_monetario_colones']/1000, alpha=0.5, s=20, c='#00d4ff')
axes[1].set_xlabel('Frecuencia (compras)')
axes[1].set_ylabel('Valor monetario (miles de colones)')
axes[1].set_title('Frecuencia vs Valor monetario')

axes[2].scatter(df['recencia_dias'], df['valor_monetario_colones']/1000, alpha=0.5, s=20, c='#7c3aed')
axes[2].set_xlabel('Recencia (dias)')
axes[2].set_ylabel('Valor monetario (miles de colones)')
axes[2].set_title('Recencia vs Valor monetario')

plt.tight_layout()
plt.suptitle('Relaciones entre variables RFM', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("Observen los scatter plots: ¿pueden intuir grupos de clientes?")
print("Esos patrones que ven a simple vista son los que el clustering formalizara.")

## 5. Paso critico: normalizacion de datos

Observen las escalas de nuestras variables:
- Recencia: 1 a ~250 dias
- Frecuencia: 1 a ~40 compras
- Valor monetario: 5,000 a ~200,000 colones

Si aplicamos clustering directamente, **el valor monetario dominaria completamente** porque sus numeros son mucho mas grandes. El algoritmo pensaria que la diferencia entre 100,000 y 150,000 colones es mucho mas importante que la diferencia entre 5 y 25 compras, simplemente porque los numeros son mas grandes.

La **normalizacion** (StandardScaler) transforma cada variable para que tenga media = 0 y desviacion estandar = 1. Asi, todas las variables contribuyen equitativamente al clustering.

> **Regla de oro:** Siempre normalizar antes de aplicar clustering (excepto si todas las variables ya estan en la misma escala).

In [ ]:
# =============================================================
# Normalizacion con StandardScaler
# =============================================================
scaler = StandardScaler()
datos_normalizados = scaler.fit_transform(df)
df_norm = pd.DataFrame(datos_normalizados, columns=df.columns)

print("Antes de normalizar:")
print(f"  Recencia   -> Media: {df['recencia_dias'].mean():.1f}, Std: {df['recencia_dias'].std():.1f}")
print(f"  Frecuencia -> Media: {df['frecuencia_compras'].mean():.1f}, Std: {df['frecuencia_compras'].std():.1f}")
print(f"  Valor      -> Media: {df['valor_monetario_colones'].mean():.0f}, Std: {df['valor_monetario_colones'].std():.0f}")

print("\nDespues de normalizar:")
print(f"  Recencia   -> Media: {df_norm['recencia_dias'].mean():.4f}, Std: {df_norm['recencia_dias'].std():.4f}")
print(f"  Frecuencia -> Media: {df_norm['frecuencia_compras'].mean():.4f}, Std: {df_norm['frecuencia_compras'].std():.4f}")
print(f"  Valor      -> Media: {df_norm['valor_monetario_colones'].mean():.4f}, Std: {df_norm['valor_monetario_colones'].std():.4f}")

print("\nAhora todas las variables tienen la misma escala. El clustering sera justo.")

## 6. Algoritmo 1: K-Means

### ¿Como funciona?

K-Means es el algoritmo de clustering mas utilizado en la industria por su simplicidad y velocidad. Funciona en 4 pasos que se repiten:

1. **Elegir K** (el numero de grupos que queremos)
2. **Colocar K centros** (llamados "centroides") en posiciones iniciales aleatorias
3. **Asignar** cada dato al centroide mas cercano
4. **Recalcular** cada centroide como el promedio de los datos asignados a el

Los pasos 3 y 4 se repiten hasta que los centroides dejan de moverse significativamente (convergencia).

### Analogia

Imaginen que quieren organizar 500 estudiantes en K mesas de estudio en una biblioteca. Colocan K mesas en posiciones aleatorias, cada estudiante va a la mesa mas cercana, luego mueven cada mesa al centro de su grupo de estudiantes. Repiten hasta que las mesas dejen de moverse.

### Pero... ¿como elegimos K?

El **metodo del codo (Elbow Method)** nos ayuda. Probamos K = 2, 3, 4, ..., 10, y medimos que tan compactos quedan los grupos (inercia). Graficamos y buscamos el "codo" donde agregar mas grupos ya no mejora significativamente.

In [ ]:
# =============================================================
# Metodo del codo para determinar el numero optimo de clusters
# =============================================================
inercias = []
rango_k = range(2, 11)

for k in rango_k:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_norm)
    inercias.append(kmeans.inertia_)

# Graficar
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rango_k, inercias, 'o-', color='#2d1b4e', linewidth=2, markersize=8)
ax.set_xlabel('Numero de clusters (K)', fontsize=12)
ax.set_ylabel('Inercia (suma de distancias al centroide)', fontsize=12)
ax.set_title('Metodo del codo: ¿cuantos clusters necesitamos?', fontsize=14, fontweight='bold')
ax.set_xticks(range(2, 11))

# Marcar el codo sugerido
ax.annotate('Codo sugerido (K=4)', xy=(4, inercias[2]), xytext=(6, inercias[2] + 100),
            arrowprops=dict(arrowstyle='->', color='#00d4ff', lw=2),
            fontsize=12, color='#00d4ff', fontweight='bold')

plt.tight_layout()
plt.show()

print("La inercia mide que tan lejos estan los puntos de su centroide.")
print("A menor inercia, mas compactos los grupos.")
print("Buscamos el 'codo': el punto donde agregar mas clusters ya no reduce mucho la inercia.")
print("\nEn este caso, K=4 parece ser el codo.")

In [ ]:
# =============================================================
# Aplicar K-Means con K=4
# =============================================================
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
etiquetas_kmeans = kmeans.fit_predict(df_norm)

# Agregar etiquetas al DataFrame original (no normalizado, para interpretar)
df['cluster_kmeans'] = etiquetas_kmeans

# Visualizar los clusters en 2D (Frecuencia vs Valor Monetario)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colores_clusters = ['#2d1b4e', '#00d4ff', '#7c3aed', '#f59e0b', '#10b981', '#ef4444', '#6366f1', '#ec4899']

# Grafico 1: Frecuencia vs Valor Monetario
for i in range(4):
    mask = etiquetas_kmeans == i
    axes[0].scatter(df.loc[mask, 'frecuencia_compras'],
                    df.loc[mask, 'valor_monetario_colones'] / 1000,
                    c=colores_clusters[i], label=f'Cluster {i}', alpha=0.6, s=30)
axes[0].set_xlabel('Frecuencia (compras)')
axes[0].set_ylabel('Valor monetario (miles de colones)')
axes[0].set_title('K-Means: Frecuencia vs Valor monetario')
axes[0].legend()

# Grafico 2: Recencia vs Frecuencia
for i in range(4):
    mask = etiquetas_kmeans == i
    axes[1].scatter(df.loc[mask, 'recencia_dias'],
                    df.loc[mask, 'frecuencia_compras'],
                    c=colores_clusters[i], label=f'Cluster {i}', alpha=0.6, s=30)
axes[1].set_xlabel('Recencia (dias)')
axes[1].set_ylabel('Frecuencia (compras)')
axes[1].set_title('K-Means: Recencia vs Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.suptitle('Resultados de K-Means con K=4', fontsize=14, fontweight='bold', y=1.02)
plt.show()

In [ ]:
# =============================================================
# Interpretar los clusters: ¿que significa cada grupo para el negocio?
# =============================================================
resumen = df.groupby('cluster_kmeans').agg(
    cantidad=('recencia_dias', 'count'),
    recencia_promedio=('recencia_dias', 'mean'),
    frecuencia_promedio=('frecuencia_compras', 'mean'),
    valor_promedio=('valor_monetario_colones', 'mean')
).round(1)

resumen['valor_promedio_miles'] = (resumen['valor_promedio'] / 1000).round(1)

print("=== Perfil de cada cluster ===\n")
print(resumen[['cantidad', 'recencia_promedio', 'frecuencia_promedio', 'valor_promedio_miles']].to_string())

print("\n\n=== Interpretacion de negocio ===\n")
print("Ahora viene la parte mas importante: NOMBRAR los clusters.")
print("Los numeros (0, 1, 2, 3) no significan nada por si solos.")
print("El equipo de negocio debe interpretar cada perfil y asignar nombres accionables.")
print("\nPor ejemplo:")
print("  - Cluster con baja recencia + alta frecuencia + alto valor = 'Champions' o 'Premium'")
print("  - Cluster con alta recencia + baja frecuencia + bajo valor = 'En riesgo' o 'Inactivos'")
print("\n¿Que nombres le pondrian ustedes a cada cluster?")

## 7. Algoritmo 2: Clustering jerarquico

### ¿Como funciona?

A diferencia de K-Means, el clustering jerarquico **no requiere que definamos K de antemano**. Construye una estructura de arbol (dendrograma) que muestra como se relacionan los datos a multiples niveles.

La variante **aglomerativa** (la mas comun) funciona asi:
1. Cada dato empieza como su propio cluster individual
2. Se fusionan los dos clusters mas similares
3. Se repite hasta que todo es un solo cluster

El resultado es un **dendrograma**: un arbol que podemos "cortar" a la altura que queramos para obtener el numero de grupos que nos convenga.

### Analogia

Piensen en un arbol genealogico: empezamos con individuos, luego agrupamos hermanos, luego familias nucleares, luego familia extendida, hasta llegar a toda la familia. El dendrograma nos deja elegir a que "nivel" queremos ver las agrupaciones.

In [ ]:
# =============================================================
# Dendrograma: visualizar la estructura jerarquica
# =============================================================

# Calculamos la matriz de enlace (usamos el metodo Ward, que minimiza varianza intra-cluster)
Z = linkage(df_norm, method='ward')

fig, ax = plt.subplots(figsize=(16, 7))
dendrogram(Z,
           truncate_mode='lastp',  # Solo mostrar los ultimos p clusters fusionados
           p=30,                    # Mostrar los ultimos 30 nodos
           leaf_rotation=90,
           leaf_font_size=9,
           color_threshold=7,       # Colores automaticos basados en distancia
           ax=ax)

ax.set_title('Dendrograma: estructura jerarquica de los clientes', fontsize=14, fontweight='bold')
ax.set_xlabel('Clientes (agrupados)', fontsize=12)
ax.set_ylabel('Distancia de fusion (Ward)', fontsize=12)

# Linea de corte sugerida
ax.axhline(y=7, color='red', linestyle='--', linewidth=2, label='Corte sugerido (4 clusters)')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("El dendrograma muestra como se van fusionando los clientes.")
print("La linea roja punteada sugiere un corte que produce 4 clusters.")
print("Si cortaramos mas abajo, tendriamos mas clusters (mas especificos).")
print("Si cortaramos mas arriba, tendriamos menos clusters (mas generales).")
print("\nEsta flexibilidad es la gran ventaja del clustering jerarquico.")

In [ ]:
# =============================================================
# Aplicar clustering jerarquico aglomerativo con 4 clusters
# =============================================================
jerarquico = AgglomerativeClustering(n_clusters=4, linkage='ward')
etiquetas_jerarquico = jerarquico.fit_predict(df_norm)

df['cluster_jerarquico'] = etiquetas_jerarquico

# Visualizar comparacion K-Means vs Jerarquico
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i in range(4):
    mask_km = df['cluster_kmeans'] == i
    axes[0].scatter(df.loc[mask_km, 'frecuencia_compras'],
                    df.loc[mask_km, 'valor_monetario_colones'] / 1000,
                    c=colores_clusters[i], label=f'Cluster {i}', alpha=0.6, s=30)
axes[0].set_title('K-Means (K=4)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Frecuencia')
axes[0].set_ylabel('Valor (miles colones)')
axes[0].legend()

for i in range(4):
    mask_jq = df['cluster_jerarquico'] == i
    axes[1].scatter(df.loc[mask_jq, 'frecuencia_compras'],
                    df.loc[mask_jq, 'valor_monetario_colones'] / 1000,
                    c=colores_clusters[i], label=f'Cluster {i}', alpha=0.6, s=30)
axes[1].set_title('Jerarquico aglomerativo (Ward, K=4)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Frecuencia')
axes[1].set_ylabel('Valor (miles colones)')
axes[1].legend()

plt.tight_layout()
plt.suptitle('Comparacion: K-Means vs Clustering jerarquico', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("Observen: ambos algoritmos producen resultados similares pero NO identicos.")
print("Los numeros de cluster pueden no coincidir (Cluster 0 en K-Means no es")
print("necesariamente Cluster 0 en Jerarquico). Lo importante es la composicion de los grupos.")

## 8. Algoritmo 3: DBSCAN

### ¿Como funciona?

DBSCAN es radicalmente diferente a K-Means y al jerarquico. En vez de buscar centros o construir arboles, agrupa puntos que estan **densamente empaquetados** y define clusters como **regiones de alta densidad separadas por regiones de baja densidad**.

Tiene dos parametros:
- **epsilon (eps):** radio maximo para considerar que dos puntos son "vecinos"
- **min_samples:** minimo de vecinos que necesita un punto para ser considerado "nucleo" de un cluster

### Ventajas unicas de DBSCAN
- **No requiere especificar K** (el numero de clusters)
- **Detecta formas arbitrarias** (no solo esferas como K-Means)
- **Identifica automaticamente ruido** (outliers) y los marca como cluster -1

### Limitacion
- Es sensible a los valores de eps y min_samples
- Si la densidad varia mucho entre clusters, puede fallar

### Cuando usarlo
Ideal cuando sospechan que hay **valores atipicos** en los datos o cuando los grupos podrian tener **formas irregulares** (no esfericas).

> **Nota avanzada:** Existe una version mejorada llamada **HDBSCAN** que maneja densidades variables. Ya esta integrado en scikit-learn y es considerado por muchos expertos como el mejor algoritmo general de clustering disponible actualmente.

In [ ]:
# =============================================================
# Primero, veamos DBSCAN con datos que demuestren su fortaleza:
# datos con formas NO esfericas (donde K-Means falla)
# =============================================================

# Generamos datos en forma de medias lunas (moons)
X_lunas, _ = make_moons(n_samples=300, noise=0.08, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- K-Means en datos tipo luna ---
km_lunas = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km = km_lunas.fit_predict(X_lunas)
axes[0].scatter(X_lunas[:, 0], X_lunas[:, 1], c=labels_km, cmap='coolwarm', alpha=0.7, s=40)
axes[0].set_title('K-Means (falla con formas irregulares)', fontsize=12, fontweight='bold')

# --- DBSCAN en datos tipo luna ---
db_lunas = DBSCAN(eps=0.2, min_samples=5)
labels_db = db_lunas.fit_predict(X_lunas)
scatter = axes[1].scatter(X_lunas[:, 0], X_lunas[:, 1], c=labels_db, cmap='coolwarm', alpha=0.7, s=40)
axes[1].set_title('DBSCAN (detecta formas arbitrarias)', fontsize=12, fontweight='bold')

# --- DBSCAN identificando ruido ---
# Agregamos outliers
X_ruido = np.vstack([X_lunas, np.random.uniform(-1.5, 2.5, (20, 2))])
db_ruido = DBSCAN(eps=0.2, min_samples=5)
labels_ruido = db_ruido.fit_predict(X_ruido)

colores_ruido = ['#2d1b4e' if l == 0 else '#00d4ff' if l == 1 else '#ff4444' for l in labels_ruido]
axes[2].scatter(X_ruido[:, 0], X_ruido[:, 1], c=colores_ruido, alpha=0.7, s=40)
axes[2].set_title('DBSCAN con ruido (rojo = outliers)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.suptitle('¿Por que necesitamos DBSCAN?', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print("Imagen izquierda: K-Means falla porque asume clusters esfericos.")
print("Imagen central: DBSCAN detecta correctamente las dos lunas.")
print("Imagen derecha: DBSCAN ademas identifica puntos de ruido (outliers) en rojo.")

In [ ]:
# =============================================================
# Ahora aplicamos DBSCAN a nuestros datos RFM de clientes
# =============================================================
dbscan = DBSCAN(eps=0.4, min_samples=10)
etiquetas_dbscan = dbscan.fit_predict(df_norm)

df['cluster_dbscan'] = etiquetas_dbscan

n_clusters_db = len(set(etiquetas_dbscan)) - (1 if -1 in etiquetas_dbscan else 0)
n_ruido = (etiquetas_dbscan == -1).sum()

print(f"DBSCAN encontro: {n_clusters_db} clusters")
print(f"Puntos clasificados como ruido (outliers): {n_ruido} ({n_ruido/len(df)*100:.1f}%)")
print(f"\nDistribucion de clusters:")
print(pd.Series(etiquetas_dbscan).value_counts().sort_index().to_string())

# Visualizar
fig, ax = plt.subplots(figsize=(10, 6))

# Ruido en gris
mask_ruido = etiquetas_dbscan == -1
ax.scatter(df.loc[mask_ruido, 'frecuencia_compras'],
           df.loc[mask_ruido, 'valor_monetario_colones'] / 1000,
           c='gray', marker='x', alpha=0.5, s=30, label='Ruido/Outliers')

# Clusters con colores
for i in range(n_clusters_db):
    mask = etiquetas_dbscan == i
    ax.scatter(df.loc[mask, 'frecuencia_compras'],
               df.loc[mask, 'valor_monetario_colones'] / 1000,
               c=colores_clusters[i % len(colores_clusters)],
               label=f'Cluster {i}', alpha=0.6, s=30)

ax.set_xlabel('Frecuencia (compras)')
ax.set_ylabel('Valor monetario (miles de colones)')
ax.set_title(f'DBSCAN: {n_clusters_db} clusters + outliers', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print("\nDBSCAN tiene una ventaja unica: identifica automaticamente clientes 'anomalos'")
print("que no encajan en ningun grupo. Esto es valioso para detectar fraude o")
print("comportamientos inusuales de compra.")
print("\nNoten que DBSCAN encontro un numero de clusters DIFERENTE al que usamos")
print("en K-Means. Esto es normal: cada algoritmo 've' la estructura de forma distinta.")
print("No hay una respuesta correcta unica; depende de lo que necesite el negocio.")

## 9. Evaluacion: ¿como sabemos si un clustering es bueno?

Este es uno de los mayores desafios del aprendizaje no supervisado: **no hay "respuesta correcta"** contra la cual comparar. A diferencia de la clasificacion (donde podemos medir accuracy, precision, recall), aqui necesitamos metricas especiales.

### Las tres metricas principales

| Metrica | Rango | ¿Mejor es...? | ¿Que mide? |
|---------|-------|----------------|------------|
| **Coeficiente de Silueta** | -1 a +1 | Mayor | Cohesion interna vs separacion entre clusters |
| **Indice Davies-Bouldin** | 0 a infinito | Menor | Ratio de dispersion intra-cluster vs distancia inter-cluster |
| **Indice Calinski-Harabasz** | 0 a infinito | Mayor | Varianza entre clusters vs varianza dentro de clusters |

### Coeficiente de Silueta: la metrica mas intuitiva

Para cada punto, mide:
- **a:** distancia promedio a los demas puntos de SU cluster (cohesion)
- **b:** distancia promedio al cluster MAS CERCANO (separacion)
- **Silueta = (b - a) / max(a, b)**

Interpretacion:
- Cerca de **+1:** el punto esta bien asignado a su cluster
- Cerca de **0:** el punto esta en la frontera entre clusters
- **Negativo:** el punto probablemente esta en el cluster equivocado

> **Hallazgo reciente (2025):** Un estudio publicado en PeerJ Computer Science demostro que el Silueta funciona bien solo para clusters esfericos. Para formas irregulares, puede dar puntajes enganosos. Se recomienda siempre usar **multiples metricas**, no una sola.

In [ ]:
# =============================================================
# Evaluacion comparativa de los tres algoritmos
# =============================================================

# Solo evaluamos puntos que no son ruido en DBSCAN
mask_no_ruido = etiquetas_dbscan != -1

resultados = {}

# K-Means
resultados['K-Means'] = {
    'Silueta': silhouette_score(df_norm, etiquetas_kmeans),
    'Davies-Bouldin': davies_bouldin_score(df_norm, etiquetas_kmeans),
    'Calinski-Harabasz': calinski_harabasz_score(df_norm, etiquetas_kmeans),
    'Num. clusters': 4
}

# Jerarquico
resultados['Jerarquico'] = {
    'Silueta': silhouette_score(df_norm, etiquetas_jerarquico),
    'Davies-Bouldin': davies_bouldin_score(df_norm, etiquetas_jerarquico),
    'Calinski-Harabasz': calinski_harabasz_score(df_norm, etiquetas_jerarquico),
    'Num. clusters': 4
}

# DBSCAN (solo puntos no-ruido)
if n_clusters_db >= 2:
    resultados['DBSCAN'] = {
        'Silueta': silhouette_score(df_norm[mask_no_ruido], etiquetas_dbscan[mask_no_ruido]),
        'Davies-Bouldin': davies_bouldin_score(df_norm[mask_no_ruido], etiquetas_dbscan[mask_no_ruido]),
        'Calinski-Harabasz': calinski_harabasz_score(df_norm[mask_no_ruido], etiquetas_dbscan[mask_no_ruido]),
        'Num. clusters': n_clusters_db
    }

df_eval = pd.DataFrame(resultados).T
print("=== Comparacion de metricas de evaluacion ===\n")
print(df_eval.round(3).to_string())
print("\n--- Guia de interpretacion ---")
print("Silueta: mayor es mejor (rango -1 a +1)")
print("Davies-Bouldin: menor es mejor (rango 0 a infinito)")
print("Calinski-Harabasz: mayor es mejor (rango 0 a infinito)")

In [ ]:
# =============================================================
# Visualizacion del Silueta por cluster (K-Means)
# =============================================================
fig, ax = plt.subplots(figsize=(10, 7))

silueta_valores = silhouette_samples(df_norm, etiquetas_kmeans)
silueta_promedio = silhouette_score(df_norm, etiquetas_kmeans)

y_lower = 10
for i in range(4):
    cluster_vals = silueta_valores[etiquetas_kmeans == i]
    cluster_vals.sort()
    size_cluster = len(cluster_vals)
    y_upper = y_lower + size_cluster

    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_vals,
                      alpha=0.7, color=colores_clusters[i],
                      label=f'Cluster {i} (n={size_cluster})')
    ax.text(-0.05, y_lower + 0.5 * size_cluster, str(i), fontsize=12, fontweight='bold')
    y_lower = y_upper + 10

ax.axvline(x=silueta_promedio, color='red', linestyle='--', linewidth=2,
           label=f'Promedio global: {silueta_promedio:.3f}')
ax.set_xlabel('Coeficiente de Silueta', fontsize=12)
ax.set_ylabel('Clientes (agrupados por cluster)', fontsize=12)
ax.set_title('Diagrama de Silueta por cluster (K-Means)', fontsize=14, fontweight='bold')
ax.legend(loc='best')

plt.tight_layout()
plt.show()

print("Cada barra horizontal representa un cliente.")
print("Si la barra es larga (cercana a 1), el cliente esta bien asignado a su cluster.")
print("Si la barra cruza al lado negativo, el cliente podria estar mejor en otro cluster.")
print(f"\nEl promedio global de Silueta es {silueta_promedio:.3f}")

## 10. Determinando el K optimo: metodo del codo + Silueta combinados

El metodo del codo que vimos antes es una guia visual, pero podemos ser mas rigurosos combinandolo con el Silueta para diferentes valores de K.

In [ ]:
# =============================================================
# Silueta para diferentes valores de K
# =============================================================
siluetas = []
rango_k = range(2, 11)

for k in rango_k:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_temp = kmeans_temp.fit_predict(df_norm)
    siluetas.append(silhouette_score(df_norm, labels_temp))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Grafico 1: Codo
axes[0].plot(rango_k, inercias, 'o-', color='#2d1b4e', linewidth=2, markersize=8)
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Metodo del codo', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(2, 11))

# Grafico 2: Silueta
axes[1].plot(rango_k, siluetas, 's-', color='#00d4ff', linewidth=2, markersize=8)
axes[1].set_xlabel('K')
axes[1].set_ylabel('Coeficiente de Silueta')
axes[1].set_title('Silueta promedio por K', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(2, 11))

# Marcar K optimo
k_optimo = list(rango_k)[np.argmax(siluetas)]
axes[1].axvline(x=k_optimo, color='red', linestyle='--', alpha=0.7,
                label=f'Mejor K={k_optimo} (Silueta={max(siluetas):.3f})')
axes[1].legend()

plt.tight_layout()
plt.suptitle('¿Cual es el K optimo?', fontsize=14, fontweight='bold', y=1.02)
plt.show()

print(f"Segun el Silueta, el K optimo es: {k_optimo}")
print(f"\nPero recuerden: la metrica es solo una guia.")
print("El K final debe tener SENTIDO DE NEGOCIO.")
print("Si K=2 da mejor Silueta pero K=4 produce segmentos mas accionables,")
print("el equipo de negocio podria preferir K=4.")

## 11. Lo mas importante: interpretacion de negocio

Los numeros y graficos que hemos visto son herramientas, no respuestas finales. El verdadero valor del clustering emerge cuando el equipo de negocio **interpreta y nombra los segmentos** para tomar decisiones accionables.

Veamos el resumen final de nuestros clusters K-Means y asignemosles nombres de negocio.

In [ ]:
# =============================================================
# Resumen ejecutivo de segmentacion
# =============================================================
resumen_final = df.groupby('cluster_kmeans').agg(
    clientes=('recencia_dias', 'count'),
    recencia_media=('recencia_dias', 'mean'),
    frecuencia_media=('frecuencia_compras', 'mean'),
    valor_medio_colones=('valor_monetario_colones', 'mean')
).round(1)

resumen_final['pct_clientes'] = (resumen_final['clientes'] / resumen_final['clientes'].sum() * 100).round(1)
resumen_final['valor_medio_miles'] = (resumen_final['valor_medio_colones'] / 1000).round(1)

# Asignar nombres basados en los perfiles
# (Los clusters pueden tener numeros diferentes en cada ejecucion,
#  por lo que asignamos nombres segun el perfil observado)
nombres = {}
for idx, row in resumen_final.iterrows():
    if row['recencia_media'] < 30 and row['frecuencia_media'] > 20 and row['valor_medio_colones'] > 100000:
        nombres[idx] = 'Champions (Premium)'
    elif row['frecuencia_media'] > 15 and row['valor_medio_colones'] < 60000:
        nombres[idx] = 'Leales (bajo valor)'
    elif row['recencia_media'] > 90:
        nombres[idx] = 'En riesgo (Inactivos)'
    else:
        nombres[idx] = 'Esporadicos (Potenciales)'

resumen_final['segmento'] = resumen_final.index.map(nombres)

print("=" * 80)
print("RESUMEN EJECUTIVO: SEGMENTACION DE CLIENTES")
print("=" * 80)

for idx, row in resumen_final.iterrows():
    print(f"\n--- {row['segmento']} ---")
    print(f"  Cantidad: {int(row['clientes'])} clientes ({row['pct_clientes']}%)")
    print(f"  Recencia promedio: {row['recencia_media']:.0f} dias")
    print(f"  Frecuencia promedio: {row['frecuencia_media']:.0f} compras")
    print(f"  Valor promedio: {row['valor_medio_miles']:.0f} mil colones")

print("\n" + "=" * 80)
print("ACCIONES SUGERIDAS POR SEGMENTO:")
print("=" * 80)
for idx, row in resumen_final.iterrows():
    seg = row['segmento']
    if 'Champions' in seg:
        print(f"\n  {seg}: Programa VIP, acceso anticipado, beneficios exclusivos")
    elif 'Leales' in seg:
        print(f"  {seg}: Upselling, bundles de mayor valor, programa de recompensas")
    elif 'Esporadicos' in seg:
        print(f"  {seg}: Campanas de reactivacion, descuentos personalizados")
    elif 'riesgo' in seg:
        print(f"  {seg}: Campana urgente de recuperacion, encuesta de satisfaccion")

In [ ]:
# =============================================================
# Visualizacion final: perfil radar (spider chart) por segmento
# =============================================================
from math import pi

categorias = ['Recencia\n(inversa)', 'Frecuencia', 'Valor\nmonetario']
N = len(categorias)

# Normalizar a escala 0-1 para el radar
vals_norm = resumen_final[['recencia_media', 'frecuencia_media', 'valor_medio_colones']].copy()

# Invertir recencia (menor recencia = mejor = valor mas alto en el radar)
vals_norm['recencia_media'] = vals_norm['recencia_media'].max() - vals_norm['recencia_media']

for col in vals_norm.columns:
    rango = vals_norm[col].max() - vals_norm[col].min()
    if rango > 0:
        vals_norm[col] = (vals_norm[col] - vals_norm[col].min()) / rango

angulos = [n / float(N) * 2 * pi for n in range(N)]
angulos += angulos[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for idx, row in vals_norm.iterrows():
    valores = row.tolist()
    valores += valores[:1]
    nombre = resumen_final.loc[idx, 'segmento']
    color = colores_clusters[idx % len(colores_clusters)]
    ax.plot(angulos, valores, 'o-', linewidth=2, label=nombre, color=color)
    ax.fill(angulos, valores, alpha=0.1, color=color)

ax.set_xticks(angulos[:-1])
ax.set_xticklabels(categorias, fontsize=11)
ax.set_title('Perfil de cada segmento de clientes', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout()
plt.show()

print("Este grafico radar permite comparar visualmente los perfiles de cada segmento.")
print("Es una herramienta excelente para presentar resultados a stakeholders no tecnicos.")

## 12. Resumen: ¿cuando usar cada algoritmo?

| Escenario | Algoritmo recomendado | Razon |
|-----------|----------------------|-------|
| Segmentacion de clientes con K conocido | **K-Means** | Rapido, escalable, interpretable |
| Explorar datos sin saber K | **Jerarquico** | El dendrograma revela estructura natural |
| Datos con formas irregulares y ruido | **DBSCAN** | Detecta formas arbitrarias y outliers |
| Datos masivos (millones de registros) | **Mini-Batch K-Means** | Eficiencia computacional |
| Densidades variables en datos reales | **HDBSCAN** | Robusto ante densidades mixtas |

### Conceptos erroneos comunes

1. **"El clustering siempre produce la misma respuesta"** — K-Means es sensible a la inicializacion; diferentes ejecuciones pueden dar resultados distintos.
2. **"Mas clusters = mejor resultado"** — No necesariamente. Usamos el codo y Silueta como guia, pero el criterio de negocio es fundamental.
3. **"El clustering encuentra LA verdad en los datos"** — Los clusters son UNA interpretacion. El juicio experto es esencial para validar y nombrar grupos.
4. **"Solo se necesita el algoritmo"** — La preparacion de datos (normalizacion, seleccion de variables) es tan importante como el algoritmo.

### Herramientas recomendadas para explorar por su cuenta

- **Visualizacion interactiva de K-Means:** [naftaliharris.com/blog/visualizing-k-means-clustering](https://www.naftaliharris.com/blog/visualizing-k-means-clustering/)
- **Visualizacion interactiva de DBSCAN:** [naftaliharris.com/blog/visualizing-dbscan-clustering](https://www.naftaliharris.com/blog/visualizing-dbscan-clustering/)
- **Orange Data Mining (sin codigo):** [orangedatamining.com](https://orangedatamining.com/)
- **Comparacion de algoritmos (scikit-learn):** [scikit-learn.org/stable/auto_examples/cluster/plot_cluster_comparison.html](https://scikit-learn.org/stable/auto_examples/cluster/plot_cluster_comparison.html)

---

## Preguntas de discusion para la sesion

1. En nuestro ejemplo, el Silueta apuntaba a un K optimo pero dijimos que el criterio de negocio puede preferir otro K. ¿En que situaciones reales el juicio empresarial deberia prevalecer sobre la metrica?

2. DBSCAN identifico ciertos clientes como "ruido". ¿Que significaria esto en un contexto real de negocios? ¿Son clientes que debemos ignorar, investigar con mas detalle, o algo diferente?

3. Si tuvieran que presentar esta segmentacion al gerente general de una empresa, ¿que informacion incluirian y cual dejarian fuera? ¿Como presentarian los resultados sin jerga tecnica?

4. La normalizacion cambio fundamentalmente los resultados del clustering. ¿Pueden pensar en un escenario donde normalizar NO seria apropiado?

5. Netflix usa mas de 1,300 clusters de recomendacion. ¿Por que creen que necesitan tantos en vez de 4 o 5 segmentos simples?

---
*Notebook preparado para Fundamentos de IA y Machine Learning — ULACIT, 1c0-2026*